In [14]:
import pandas as pd
import os
from datetime import datetime, timezone

RAW_FILE   = "raw_data/databridge_squad19_sintetico.csv"

df = pd.read_csv('../raw_data/databridge_squad19_sintetico.csv', dtype=str)

In [15]:
ingested_at = datetime.now(timezone.utc).isoformat()

print(f"Linhas  : {len(df):,}")
print(f"Colunas : {df.shape[1]}")
print(f"Arquivo : {os.path.basename(RAW_FILE)}")

Linhas  : 30,000
Colunas : 62
Arquivo : databridge_squad19_sintetico.csv


In [18]:
# Primeiras linhas
df.head()

,application_id,source_system,ocr_engine,doc_type,page_count,file_size_kb,image_blur,skew_angle_deg,document_image_quality,ocr_confidence,...,credit_requested_value,income_declared,tenure_months,collateral_type,ltv,pd_model_score,final_decision,reason_code,requires_human_review,default_12m
0,APP-20260000001,OCR_PDF,azure_ocr,EXTRATO,1,1421.0,0.14,-1.35,0.821,0.562,...,10898.86,53421.8,210,IMOVEL,0.67,0.297,APPROVE,OK,1,0
1,APP-20260000002,OCR_PDF,tesseract,RG,9,1556.0,0.016,0.5,0.968,0.689,...,19605.0,22433.06,125,IMOVEL,1.63,0.346,REVIEW,HIGH_RISK,1,0
2,APP-20260000003,EMAIL_ATTACH,google_vision,EXTRATO,8,982.0,0.338,1.8,0.622,0.83,...,26290.89,28138.01,152,VEICULO,0.67,0.223,APPROVE,OK,0,1
3,APP-20260000004,MOBILE_PHOTO,google_vision,CONTRATO,5,359.0,0.56,-0.66,0.43,0.649,...,59538.91,41331.41,213,CPR,2.0,0.33,REVIEW,HIGH_RISK,1,1
4,APP-20260000005,MOBILE_PHOTO,google_vision,EXTRATO,5,1559.0,0.169,1.6,0.787,0.673,...,6890.87,25293.87,99,SEM_GARANTIA,0.5,0.338,APPROVE,OK,0,0


In [17]:
df.dtypes.value_counts()

str    62
Name: count, dtype: int64

In [19]:
# Contagem de nulos por coluna (como string, nulos aparecem como NaN)
nulls = df.isnull().sum()
nulls[nulls > 0].rename("null_count").to_frame()

,null_count
ocr_confidence,1440
duplicate_group_id,29700


In [20]:
print("── file_size_kb (drift com sufixo KB) ──")
print(df[df["file_size_kb"].str.contains("KB", na=False)]["file_size_kb"].head(5).tolist())

print("\n── raw_date_text (ruído OCR) ──")
print(df["raw_date_text"].head(10).tolist())

print("\n── raw_amount_text (formatos mistos) ──")
print(df["raw_amount_text"].head(10).tolist())

── file_size_kb (drift com sufixo KB) ──
['424 KB', '842 KB', '896 KB', '495 KB', '925 KB']

── raw_date_text (ruído OCR) ──
['19/06/2024', '2025-0B-25', '17/09/2025', '19/03/2025', '2O25-1I-02', '2024-I1-*25', '02/10/202S', '01/05/2025', '25/04/2025', '29/01/2025']

── raw_amount_text (formatos mistos) ──
['#10,238.20', 'R$ 21.253,26', 'R$ 21.342,59', '46,439.06', '5,805.61', 'R$ 1*3.480,10', '9,846.44', 'R$ 7.542,3S', 'R$ 6.703,34', 'R$ 2.446,*31']


In [21]:
total_rows  = len(df)
total_cols  = df.shape[1]
total_cells = total_rows * total_cols
null_cells  = df.isnull().sum().sum()
kb_drift    = df["file_size_kb"].str.contains("KB", na=False).sum()
dup_filled  = df["duplicate_group_id"].notna().sum()

print("=" * 45)
print("RELATÓRIO BRONZE — DataBridge Squad19")
print("=" * 45)
print(f"Total de linhas          : {total_rows:,}")
print(f"Total de colunas         : {total_cols}")
print(f"Células nulas            : {null_cells:,} ({null_cells/total_cells:.2%})")
print(f"file_size_kb com 'KB'    : {kb_drift:,} ({kb_drift/total_rows:.2%})")
print(f"Registros duplicados     : {dup_filled:,} ({dup_filled/total_rows:.2%})")
print(f"Ingestão em              : {ingested_at}")
print("=" * 45)

RELATÓRIO BRONZE — DataBridge Squad19
Total de linhas          : 30,000
Total de colunas         : 62
Células nulas            : 31,140 (1.67%)
file_size_kb com 'KB'    : 887 (2.96%)
Registros duplicados     : 300 (1.00%)
Ingestão em              : 2026-03-30T22:51:28.711697+00:00


In [27]:
OUT_DIR    = "../bronze_layer"
OUT_FILE   = f"{OUT_DIR}/bronze_databridge.csv"
df.to_csv(OUT_FILE, index=False)
size_mb = os.path.getsize(OUT_FILE) / 1024 / 1024
print(f"✅ Bronze salvo em: {OUT_FILE}")
print(f"   Tamanho        : {size_mb:.2f} MB")
print(f"   Linhas         : {len(df):,}")
print(f"   Colunas        : {df.shape[1]}")

✅ Bronze salvo em: ../bronze_layer/bronze_databridge.csv
   Tamanho        : 15.58 MB
   Linhas         : 30,000
   Colunas        : 62
